In [ ]:
# 1. IMPORT LIBRARIES

import numpy as np
from PIL import Image 

import torch 
import torch.nn as nn 
import torch.nn.functional as F 
import torch.optim as optim 

import torchvision
import torchvision.transforms as transforms 


In [ ]:
# 2. DATASET AND IMAGE PROCESSING

transform = transforms.Compose ([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        
])


In [11]:
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, transform=transform, download=True)
test_data = torchvision.datasets.CIFAR10(root='./data', train=False, transform=transform, download=True)

train_loader = torch.utils.data.DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=32, shuffle=True, num_workers=2)


In [18]:
image, label = train_data[0]

In [19]:
image.size()

torch.Size([3, 32, 32])

In [16]:
class_names = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

In [ ]:
# 3. CONVOLUTIONAL NEURAL NETWORK ARCHITECTURE

class NeuralNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 12, 5) # (12, 28, 28) = new shape 
        self.pool = nn.MaxPool2d(2, 2) # (12, 14, 14) 
        self.conv2 = nn.Conv2d(12, 24, 5) # (24, 10, 10)
        self.fc1 = nn.Linear(24*5*5, 120) 
        self.fc2 = nn.Linear(120, 84) 
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x) : 
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x,1) 
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x 
    

In [ ]:
# 4. MODEL TRAINING

# Create your CNN
net = NeuralNet()

# Load your original trained weights
net.load_state_dict(
    torch.load(
        "trained_net.pth",
        map_location="cpu",
        weights_only=True
    )
)

# Put the model into testing mode
net.eval()

print("Trained model loaded successfully!")



Original trained model loaded!


In [41]:
def predict_image(image_path):

    image = Image.open(image_path).convert("RGB")
    image = image.resize((32, 32))
    image = transform(image).unsqueeze(0)

    net.eval()

    with torch.no_grad():
        outputs = net(image)
        probabilities = torch.softmax(outputs, dim=1)

        top_probabilities, top_indices = torch.topk(probabilities, 3, dim=1)

    print("Image:", image_path)
    print("Top 3 predictions:")

    for probability, index in zip(top_probabilities[0], top_indices[0]):
        print(
            f"{class_names[index.item()]}: "
            f"{probability.item() * 100:.2f}%"
        )

In [26]:
loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

In [ ]:
for epoch in range(30): 
    print(f'Training epoch {epoch}...')

    running_loss = 0.0

    for i, data in enumerate(train_loader): 
        inputs, labels = data 

        optimizer.zero_grad()

        outputs = net(inputs) 

        loss = loss_function(outputs, labels) 
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Loss: {running_loss / len(train_loader): .4f}')

Training epoch 0...


Loss:  2.1802
Training epoch 1...
Loss:  1.7582
Training epoch 2...
Loss:  1.5296
Training epoch 3...
Loss:  1.4000
Training epoch 4...
Loss:  1.3046
Training epoch 5...
Loss:  1.2256
Training epoch 6...
Loss:  1.1595
Training epoch 7...
Loss:  1.1000
Training epoch 8...
Loss:  1.0506
Training epoch 9...
Loss:  1.0049
Training epoch 10...
Loss:  0.9599
Training epoch 11...
Loss:  0.9226
Training epoch 12...
Loss:  0.8881
Training epoch 13...
Loss:  0.8573
Training epoch 14...
Loss:  0.8268
Training epoch 15...
Loss:  0.7925
Training epoch 16...
Loss:  0.7702
Training epoch 17...
Loss:  0.7405
Training epoch 18...
Loss:  0.7138
Training epoch 19...
Loss:  0.6877
Training epoch 20...
Loss:  0.6633
Training epoch 21...
Loss:  0.6387
Training epoch 22...
Loss:  0.6188
Training epoch 23...
Loss:  0.5924
Training epoch 24...
Loss:  0.5734
Training epoch 25...
Loss:  0.5503
Training epoch 26...
Loss:  0.5302
Training epoch 27...
Loss:  0.5107
Training epoch 28...
Loss:  0.4909
Training epoch 

In [ ]:
# 5. SAVE TRAINED MODEL

torch.save(net.state_dict(), 'trained_net.pth')

In [22]:
net = NeuralNet()
net.load_state_dict(torch.load('trained_net.pth'))

<All keys matched successfully>

In [ ]:
# 6. EVALUATE MODEL ACCURACY

correct = 0
total = 0 

net.eval() 

with torch.no_grad(): 
    for data in test_loader: 
        images, labels = data
        outputs = net(images) 
        _, predicted = torch.max(outputs, 1) 
        total += labels.size(0) 
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total 

print(f'Accuracy: {accuracy}%') 



Accuracy: 67.54%


In [ ]:
# 7. TEST EXTERNAL IMAGES

from pathlib import Path

image_folder = Path("test_images")

for image_path in image_folder.iterdir():

    if image_path.suffix.lower() in [".jpg", ".jpeg", ".png", ".jfif"]:

        predict_image(str(image_path))
        print("----------------------")

Image: test_images\car1.jfif
Top 3 predictions:
frog: 10.94%
horse: 10.83%
car: 10.68%
----------------------
Image: test_images\car2.jfif
Top 3 predictions:
frog: 10.97%
horse: 10.70%
car: 10.67%
----------------------
Image: test_images\dog1.jfif
Top 3 predictions:
frog: 10.89%
car: 10.82%
horse: 10.80%
----------------------
Image: test_images\frog.jfif
Top 3 predictions:
frog: 10.86%
car: 10.86%
horse: 10.85%
----------------------
Image: test_images\rosie.jpeg
Top 3 predictions:
frog: 11.00%
car: 10.82%
horse: 10.80%
----------------------
